# Exercise 4: Implement Factory Pattern for Data Transformations

##### **Objective:**
+ Practice Factory Pattern for scalable creation of transformation objects.
+ Apply abstraction and composition.
##### **Concept:** Instead of manually creating objects, use a factory that decides what transformation to apply.
##### **Instructions:**
1.  Create an abstract class DataTransform with a method apply(df).
2.  Implement subclasses:
    + NormalizeColumns
    + RemoveDuplicates
    + StandardizeText
3.  Create a TransformFactory that returns transformation objects based on string input.

In [1]:
from abc import ABC, abstractmethod
import pandas as pd

In [2]:
# 1. Abstract Class
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [3]:
# 2. Subclasses
class NormalizeColumns(DataTransform):
    """Makes column names lowercase and replaces spaces with underscores."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
        return df

In [4]:
class RemoveDuplicates(DataTransform):
    """Removes duplicate rows from the DataFrame."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)

In [5]:
class StandardizeText(DataTransform):
    """Trims extra spaces and converts text values to lowercase."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        text_cols = df.select_dtypes(include=["object", "string"]).columns
        for col in text_cols:
            df[col] = df[col].astype(str).str.strip().str.lower()
        return df

In [6]:
# 3. Factory Class
class TransformFactory:
    @staticmethod
    def create_transform(transform_type: str) -> DataTransform:
        transforms = {
            "normalize_columns": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        key = transform_type.strip().lower()
        if key not in transforms:
            raise ValueError(f"Unknown transformation: '{transform_type}'")
        return transforms[key]()

In [7]:
df = pd.read_csv("sample_data.csv")

# Apply transformations using string names
pipeline = ["normalize_columns", "standardize_text", "remove_duplicates"]

for step in pipeline:
    transformer = TransformFactory.create_transform(step)
    df = transformer.apply(df)

print(df)

   id     name   age  height_cm  weight_kg         city  score
0   1    alice  29.0      165.0       68.0     new york   85.0
1   2      bob   NaN      172.0        NaN  los angeles   90.0
2   3  charlie  35.0      168.0       72.0      chicago    NaN
3   4    david   NaN        NaN       80.0      houston   75.0
4   5      eva  27.0      160.0       55.0     new york   88.0
5   6    frank  30.0      175.0       85.0  los angeles    NaN
6   7    grace   NaN      162.0       60.0      chicago   92.0
